In [1]:
from pathlib import Path
from math import ceil
from random import shuffle
from shutil import copy2
from PIL import Image

old_dir, new_dir, output = map(Path, ['Old Textures', 'Modern Textures', 'Output'])
size = (224, 224)
renames = {
    'broken_elytra.png': 'elytra_broken.png',
    'cactus_green.png': 'green_dye.png',
    'dandelion_yellow.png': 'yellow_dye.png',
    'rose_red.png': 'red_dye.png',
    'scute.png': 'turtle_scute.png',
    'sign.png': 'oak_sign.png',
}

In [2]:
def pixels(image):
    return [pixel if pixel[3] else (0, 0, 0, 0) for pixel in image.get_flattened_data()]

pairs, excluded = [], 0
for path in sorted(old_dir.glob('*.png')):
    modern = new_dir / renames.get(path.name, path.name)
    if modern.exists():
        old = Image.open(path).convert('RGBA')
        new = Image.open(modern).convert('RGBA')
        if pixels(old) == pixels(new):
            excluded += 1
        else:
            pairs.append((modern.name, old, new))
print(f'{len(pairs)} pairs; {excluded} identical pairs excluded')

304 pairs; 44 identical pairs excluded


In [3]:
for folder in ['old', 'new', 'Train/Old', 'Train/New', 'Test/Old', 'Test/New']:
    destination = output / folder
    destination.mkdir(parents=True, exist_ok=True)
    for path in destination.glob('*.png'):
        path.unlink()

for name, old, new in pairs:
    old.resize(size, Image.Resampling.NEAREST).save(output / 'old' / name)
    new.resize(size, Image.Resampling.NEAREST).save(output / 'new' / name)

shuffle(pairs)
test_count = ceil(len(pairs) * 0.1)
for split, items in [('Test', pairs[:test_count]), ('Train', pairs[test_count:])]:
    for name, _, _ in items:
        for side in ['Old', 'New']:
            copy2(output / side.lower() / name, output / split / side / name)
print(f'Train: {len(pairs) - test_count} pairs; Test: {test_count} pairs; size: {size}')

Train: 273 pairs; Test: 31 pairs; size: (224, 224)
